# Hiring Pipeline Workflow

This workflow builds an **AI-powered hiring pipeline** that evaluates
candidates against a job description through five specialist agents:

1. **Resume Screener** -- Initial qualification check (pass/fail gate)
2. **Skills Assessor** -- Detailed skills matrix evaluation
3. **Culture Fit Evaluator** -- Red/green flag analysis
4. **Compensation Estimator** -- Market-rate salary recommendation
5. **Recommendation Agent** -- Final hiring recommendation with composite score

### Scoring Thresholds (Calibration)

Two thresholds drive the pipeline. Treat them as knobs to tune against your own offer-acceptance data, not universal truths:

| Threshold | Default | Meaning |
|---|---|---|
| Early-exit gate | `match_score < 0.5` | Resume screener rejects candidates below the "meets minimum requirements" line. |
| Composite verdict | `>= 0.85` STRONG HIRE, `>= 0.70` HIRE, `>= 0.50` BORDERLINE, else NOT RECOMMENDED | Unweighted mean of screening + 3 skills sub-scores + culture. |

If your pipeline produces too many false positives at interview, raise the early-exit gate. If it produces too few interviews, lower it.

### PII Note

The example candidates below are fully synthetic — Candidate A, B, C. No real names, emails, or employers. Always anonymize candidate data before feeding it to an LLM unless you have explicit consent and a documented data-handling policy. This is the ethical baseline, and it also avoids logging PII in model-provider telemetry.

## Step 1: Install Packages

In [ ]:
%pip install python-dotenv azure-ai-projects==2.2.0 azure-identity

## Step 2: Configure Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)
load_dotenv(override=True)

project_url = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

print("Project URL:", project_url)
print("Deployment:", deployment_name)

## Step 3: Create the Foundry Client & Helpers

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, WorkflowAgentDefinition
from azure.identity import DefaultAzureCredential

foundry_client = AIProjectClient(
    endpoint=project_url,
    credential=DefaultAzureCredential(),
    allow_preview=True,  # workflows are a preview feature; required to register WorkflowAgentDefinition
)

oai = foundry_client.get_openai_client()


def register_agent(name: str, instructions: str, description: str = ""):
    """Register a prompt agent in Foundry."""
    result = foundry_client.agents.create_version(
        agent_name=name,
        definition=PromptAgentDefinition(
            model=deployment_name,
            instructions=instructions,
        ),
        description=description,
    )
    print(f"Registered agent: {result.name} (version {result.version})")
    return result


def register_workflow(name: str, yaml_str: str, description: str = ""):
    """Register a workflow in Foundry."""
    result = foundry_client.agents.create_version(
        agent_name=name,
        definition=WorkflowAgentDefinition(workflow=yaml_str),
        description=description,
    )
    print(f"Registered workflow: {result.name} (version {result.version})")
    return result


def invoke_agent(agent_name: str, user_input: str):
    """Create a conversation and invoke a registered agent."""
    session = oai.conversations.create()
    response = oai.responses.create(
        conversation=session.id,
        input=user_input,
        extra_body={"agent_reference": {"name": agent_name, "type": "agent_reference"}},
    )
    return response.output_text


print("Foundry client ready.")

## Step 4: Register the Agents

We register five agents for the hiring pipeline.

In [ ]:
register_agent(
    name="resume-screener",
    instructions="""You are a veteran recruiter with 15 years of experience. Evaluate a candidate's resume against the provided job description.

Return ONLY a JSON object with:
- "qualified": boolean (true if the candidate meets minimum requirements)
- "match_score": float 0.0-1.0 (overall match quality)
- "gaps": array of strings listing missing qualifications
- "strengths": array of strings listing strong matches
- "summary": one-sentence assessment

Scoring guide:
- 0.0-0.3: Clearly unqualified, major gaps
- 0.3-0.5: Below minimum requirements
- 0.5-0.7: Meets minimum, some gaps
- 0.7-0.85: Strong candidate
- 0.85-1.0: Exceptional match

Cite specific evidence from the resume for each strength and gap.
Return ONLY valid JSON.""",
    description="Screens resumes against job descriptions with match scoring.",
)

In [ ]:
register_agent(
    name="skills-assessor",
    instructions="""You are a senior hiring manager. Decompose the job description into discrete skills and evaluate the candidate against each one.

Return ONLY a JSON object with:
- "technical_score": float 0.0-1.0
- "experience_score": float 0.0-1.0
- "education_score": float 0.0-1.0
- "skills_matrix": array of objects, each with:
  - "skill": string (the skill name)
  - "required_level": one of "beginner", "intermediate", "advanced", "expert"
  - "candidate_level": one of "none", "beginner", "intermediate", "advanced", "expert"
  - "gap": string describing the gap (or "none" if met)

Return ONLY valid JSON.""",
    description="Produces detailed skills assessment with proficiency matrix.",
)

register_agent(
    name="culture-evaluator",
    instructions="""You are an organizational psychologist evaluating a candidate's cultural fit.

Analyze the resume for red flags and green flags:
- Red flags: unexplained gaps >6 months, 3+ short tenures (<1 year), inconsistencies
- Green flags: progressive responsibility, long tenure, mentoring, community involvement

Return ONLY a JSON object with:
- "culture_score": float 0.0-1.0
- "red_flags": array of strings
- "green_flags": array of strings
- "team_compatibility": string (one-paragraph assessment)

Red flags are signals for discussion, not automatic disqualifiers.
Return ONLY valid JSON.""",
    description="Evaluates cultural fit with red/green flag analysis.",
)

register_agent(
    name="compensation-estimator",
    instructions="""You are a compensation analyst. Based on the candidate's qualifications, experience, and the job requirements, determine appropriate compensation.

Return ONLY a JSON object with:
- "min_salary": integer (annual USD)
- "max_salary": integer (annual USD)
- "recommended_salary": integer (annual USD)
- "market_percentile": integer (0-100, where 50 is market median)
- "justification": string explaining the recommendation

Adjust up for strong qualifications, down for gaps. All figures in annual USD.
Return ONLY valid JSON.""",
    description="Estimates compensation range based on candidate qualifications.",
)

register_agent(
    name="hiring-recommender",
    instructions="""You are a senior hiring committee advisor. You receive all prior evaluations for a candidate.

If you only receive screening data (no skills assessment, culture evaluation, or compensation estimate), produce a brief rejection summary explaining why the candidate did not pass initial screening, instead of the full recommendation report.

Otherwise, produce a structured hiring recommendation in markdown with these sections:

## Executive Summary
One paragraph with the final recommendation.

## Strengths
Bullet list of top strengths.

## Risks & Concerns
Bullet list of risks with mitigation suggestions.

## Skills Assessment Summary
Key findings from the skills evaluation.

## Culture Fit
Summary of cultural alignment.

## Compensation Recommendation
Recommended salary and justification.

## Interview Topics
3-5 specific topics to explore in an interview.

## Final Verdict
State one of: STRONG HIRE, HIRE, BORDERLINE, or NOT RECOMMENDED.
Include the composite score and a brief justification.""",
    description="Synthesizes all evaluations into a final hiring recommendation.",
)

## Step 5: Define and Register the Workflow

The workflow uses a `ConditionGroup` for the early-exit gate, and collaborates over one shared conversation:

- The screener's JSON is bound to a `responseObject` (`Local.ScreenResult`) so the gate can read `match_score` directly. The same step sets `autoSend: true`, so the screening result is also posted into the conversation.
- Every agent runs on the same `System.ConversationId` with `autoSend: true`. The skills, culture, and compensation agents each assess the candidate independently and post their JSON back.
- The recommender reads the screening, skills, culture, and compensation results straight from the conversation — no variable is threaded from one agent's output into the next agent's input — and produces the composite score and final verdict.

In [ ]:
WORKFLOW_YAML = """
kind: workflow
name: hiring-pipeline
description: Five-stage hiring evaluation with an early-exit gate; specialists collaborate over a shared conversation.
trigger:
  kind: OnConversationStart
  id: trigger_start
  actions:
    - kind: SetVariable
      id: init_source
      variable: Local.Source
      value: =System.LastMessageText

    - kind: InvokeAzureAgent
      id: invoke_screener
      agent:
        name: resume-screener
        conversationId: =System.ConversationId
      input:
        messages: =UserMessage(Local.Source)
      output:
        responseObject: Local.ScreenResult
      autoSend: true

    - kind: ConditionGroup
      id: screening_gate
      conditions:
        - id: reject_unqualified
          condition: =Local.ScreenResult.match_score < 0.5
          actions:
            - kind: InvokeAzureAgent
              id: invoke_rejection
              agent:
                name: hiring-recommender
                conversationId: =System.ConversationId
              input:
                messages: '=UserMessage("This candidate did not pass initial screening (see the screening result already in this conversation). Emit the rejection-summary variant only.")'
              output:
                messages: Local.RecommenderOutput
              autoSend: true
      elseActions:
        - kind: InvokeAzureAgent
          id: invoke_skills
          agent:
            name: skills-assessor
            conversationId: =System.ConversationId
          input:
            messages: =UserMessage(Local.Source)
          output:
            messages: Local.SkillsResult
          autoSend: true

        - kind: InvokeAzureAgent
          id: invoke_culture
          agent:
            name: culture-evaluator
            conversationId: =System.ConversationId
          input:
            messages: =UserMessage(Local.Source)
          output:
            messages: Local.CultureResult
          autoSend: true

        - kind: InvokeAzureAgent
          id: invoke_compensation
          agent:
            name: compensation-estimator
            conversationId: =System.ConversationId
          input:
            messages: =UserMessage(Local.Source)
          output:
            messages: Local.CompResult
          autoSend: true

        - kind: InvokeAzureAgent
          id: invoke_recommender
          agent:
            name: hiring-recommender
            conversationId: =System.ConversationId
          input:
            messages: '=UserMessage("Using the job and candidate plus all assessments already in this conversation (screening, skills, culture, compensation), produce the full hiring recommendation, including the composite score and final verdict.")'
          output:
            messages: Local.RecommenderOutput
          autoSend: true

    - kind: EndConversation
      id: end_conversation
""".strip()

register_workflow(
    name="hiring-pipeline",
    yaml_str=WORKFLOW_YAML,
    description="Five-stage hiring pipeline with an early-exit gate; specialists collaborate over a shared conversation.",
)

## Step 6: Verify the Workflow Executes

In [ ]:
session = oai.conversations.create()
response = oai.responses.create(
    conversation=session.id,
    input="Job: Senior Backend Engineer (Python, 5+ yrs)\n\nCandidate: 7 years Python, Go expert, Kubernetes certified",
    extra_body={"agent_reference": {"name": "hiring-pipeline", "type": "agent_reference"}},
)

print("Workflow execution trace:")
try:
    for action in response.output:
        print(f"  Action: {action.action_id} | Kind: {action.kind} | Status: {action.status}")
    print(f"\nAll {len(response.output)} actions completed successfully!")
except AttributeError:
    # Response format may vary — print raw output if trace attributes are not available
    print("Workflow response:")
    for item in response.output:
        print(f"  {item}")

## Step 7: Test the Full Pipeline

We test with three candidates of varying strength.

### Job Description

In [ ]:
JOB_DESCRIPTION = """SENIOR BACKEND ENGINEER

Requirements:
- 5+ years of professional backend development experience
- Expert-level Python and/or Go
- Experience with microservices architecture
- Strong PostgreSQL and database design skills
- Experience with message queues (Kafka, RabbitMQ)
- Kubernetes and container orchestration
- CI/CD pipeline experience
- Strong system design and API design skills

Nice to have:
- Open-source contributions
- Conference speaking or technical writing
- Team lead or mentoring experience"""

### Candidate 1: Strong (Expected: STRONG HIRE)

In [ ]:
import json
import re


def strip_code_fences(text):
    """Strip markdown code fences if the LLM wraps its JSON response."""
    text = re.sub(r'^```(?:json)?\s*\n?', '', text.strip())
    text = re.sub(r'\n?```\s*$', '', text)
    return text


def verdict_from_composite(composite: float) -> str:
    """Map composite score to the four-tier verdict used by the recommender agent."""
    if composite >= 0.85:
        return "STRONG HIRE"
    if composite >= 0.70:
        return "HIRE"
    if composite >= 0.50:
        return "BORDERLINE"
    return "NOT RECOMMENDED"


def run_hiring_pipeline(job_desc: str, resume: str):
    """Run the full hiring evaluation pipeline."""
    full_input = f"JOB DESCRIPTION:\n{job_desc}\n\nCANDIDATE RESUME:\n{resume}"

    # Stage 1: Screening
    print("=== Stage 1: Resume Screening ===")
    screen_json = invoke_agent("resume-screener", full_input)
    screen = json.loads(strip_code_fences(screen_json))
    print(f"  Qualified: {screen['qualified']}")
    print(f"  Match Score: {screen['match_score']}")
    print(f"  Summary: {screen['summary']}")

    if screen["match_score"] < 0.5:
        print(f"\n  EARLY EXIT: Score {screen['match_score']} < 0.5 threshold")
        print(f"  Gaps: {', '.join(screen['gaps'])}")
        print("  Verdict: NOT RECOMMENDED")
        return

    # Stage 2: Skills Assessment
    print("\n=== Stage 2: Skills Assessment ===")
    skills_json = invoke_agent("skills-assessor", full_input)
    skills = json.loads(strip_code_fences(skills_json))
    print(f"  Technical: {skills['technical_score']}")
    print(f"  Experience: {skills['experience_score']}")
    print(f"  Education: {skills['education_score']}")
    print(f"  Skills evaluated: {len(skills['skills_matrix'])}")

    # Stage 3: Culture Fit
    print("\n=== Stage 3: Culture Fit ===")
    culture_json = invoke_agent("culture-evaluator", full_input)
    culture = json.loads(strip_code_fences(culture_json))
    print(f"  Culture Score: {culture['culture_score']}")
    print(f"  Green Flags: {len(culture['green_flags'])}")
    print(f"  Red Flags: {len(culture['red_flags'])}")

    # Composite score (unweighted mean of the five signals)
    composite = (
        screen["match_score"]
        + skills["technical_score"]
        + skills["experience_score"]
        + skills["education_score"]
        + culture["culture_score"]
    ) / 5
    print(f"\n  Composite Score: {composite:.2f}")
    print(f"  Verdict: {verdict_from_composite(composite)}")

    # Stage 4: Compensation
    print("\n=== Stage 4: Compensation Estimate ===")
    comp_json = invoke_agent("compensation-estimator", full_input)
    comp = json.loads(strip_code_fences(comp_json))
    print(f"  Range: ${comp['min_salary']:,} - ${comp['max_salary']:,}")
    print(f"  Recommended: ${comp['recommended_salary']:,}")
    print(f"  Market Percentile: {comp['market_percentile']}th")

    # Stage 5: Final Recommendation
    print("\n=== Stage 5: Final Recommendation ===")
    all_data = (
        f"{full_input}\n\n"
        f"SCREENING: {json.dumps(screen)}\n\n"
        f"SKILLS: {json.dumps(skills)}\n\n"
        f"CULTURE: {json.dumps(culture)}\n\n"
        f"COMPENSATION: {json.dumps(comp)}\n\n"
        f"COMPOSITE SCORE: {composite:.2f}"
    )
    recommendation = invoke_agent("hiring-recommender", all_data)
    print(recommendation)


STRONG_CANDIDATE = """CANDIDATE A

EXPERIENCE:
- Senior Backend Engineer (2021-Present, 3 years)
  Led microservices migration from monolith. Python, Go, PostgreSQL, Kafka.
  Reduced API latency by 40%. Mentored 3 junior engineers.

- Backend Developer (2019-2021, 2 years)
  Built real-time data pipeline processing 100K events/sec with Python and Kafka.
  Designed PostgreSQL schema for multi-tenant SaaS platform.

- Junior Developer (2017-2019, 2 years)
  Full-stack Python/Django development. Introduced CI/CD with GitHub Actions.

EDUCATION:
- B.S. Computer Science (2017)

CERTIFICATIONS:
- Certified Kubernetes Administrator (CKA)
- AWS Solutions Architect Associate

OPEN SOURCE:
- Core contributor to a Go microservice framework (2.1K stars)
- Conference speaker: "Building Resilient Microservices"

SKILLS:
Python (expert), Go (expert), PostgreSQL (advanced), Kafka (advanced),
Kubernetes (advanced), Docker, gRPC, REST APIs, System Design"""

run_hiring_pipeline(JOB_DESCRIPTION, STRONG_CANDIDATE)

### Candidate 2: Moderate with Red Flags (Expected: BORDERLINE)

In [ ]:
MODERATE_CANDIDATE = """CANDIDATE B

EXPERIENCE:
- Backend Developer (2024-Present, 6 months)
  Python microservices development. PostgreSQL.

- Software Engineer (2023-2024, 8 months)
  Left due to leadership disagreements. Python, Django, REST APIs.

- Developer (2022-2023, 10 months)
  Backend services in Python. Basic Kubernetes usage.

  [Gap: 6 months - personal reasons]

- Junior Developer (2020-2022, 1.5 years)
  Python web development with Flask. MySQL databases.

  [Gap: 4 months]

- Intern (2019-2020, 6 months)
  Learned Python basics and SQL.

EDUCATION:
- B.S. Information Technology (2019)

SKILLS:
Python (advanced), Django (intermediate), Flask (intermediate),
PostgreSQL (intermediate), MySQL (intermediate), Docker (beginner),
Kubernetes (beginner), REST APIs (intermediate)"""

run_hiring_pipeline(JOB_DESCRIPTION, MODERATE_CANDIDATE)

### Candidate 3: Weak (Expected: EARLY EXIT)

In [ ]:
WEAK_CANDIDATE = """CANDIDATE C

EXPERIENCE:
- IT Help Desk Technician (2023-Present, 1.5 years)
  Resolve hardware/software tickets. Reset passwords. Install software.

EDUCATION:
- A.S. Information Technology (2023)

CERTIFICATIONS:
- CompTIA A+

SKILLS:
Windows administration, basic networking, HTML/CSS (beginner),
Java (beginner, coursework only), learning Python via online tutorials"""

run_hiring_pipeline(JOB_DESCRIPTION, WEAK_CANDIDATE)